# Exercise 06 Convert CloudCompare exports for QGIS

Use this notebook for step 2.4. It changes column names so QGIS can use
your CloudCompare results. It preserves every data row and numerical
value, including missing values; it does not calculate distances,
correct alignment, or apply a detection threshold.

No local Python installation or separate script upload is needed.
Sign in to Google Colab, click **Connect**, then run the three code cells
below in order using the triangle beside each cell. Use the default
CPU runtime; no GPU is needed. Your selected files are uploaded to the
Colab runtime. This notebook does not request access to Google Drive.

Before starting, export **m3c2.asc** and **c2c.asc** from CloudCompare:
choose ASCII cloud and include the column-name header. Comma, space, tab, or semicolon separators, spaces or underscores in field names, and an optional
point-count header are accepted. Keep coordinates and scalar fields. Export the
M3C2 output cloud for the first file and the June cloud carrying
C2C absolute distances for the second.

## 1 Upload your two exports

Run the cell and click **Choose Files**. Select both **m3c2.asc** and
**c2c.asc**, then wait for both uploads to finish. Do not upload the LAS
files, the companion ZIP, or a Python script. If either filename is
different, rename your export before uploading.

In [ ]:
from google.colab import files

uploaded = files.upload()
required_files = ("m3c2.asc", "c2c.asc")
missing = [name for name in required_files if name not in uploaded]
if missing:
    raise ValueError(
        "Missing " + ", ".join(missing)
        + ". Run this cell again and choose both exports."
    )
print("Both exports uploaded. Run step 2 next.")

## 2 Convert the column names

Run the cell below. A successful conversion prints the number of rows
preserved in each CSV. An error about missing fields means you should
check which cloud you exported and whether you included scalar fields
and a column-name header. Do not continue to step 3 after an error.

Each run creates a separate output folder, preserving earlier CSVs.

In [ ]:
import csv
import importlib.util
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

archive_path = None
if "uploaded" not in globals():
    raise ValueError("Run step 1 and upload both exports first.")
output_folder = Path(tempfile.mkdtemp(prefix="exercise06_conversion_"))
helper_path = output_folder / "normalize_export.py"

helper_path.write_text('"""Convert a CloudCompare ASCII export to a QGIS CSV.\n\nUsage: python3 normalize_export.py input.asc output.csv\nPreserves every row and every numerical string, including nan. Renames the\nCloudCompare scalar fields; no filtering, bias correction, or interpolation.\n"""\nfrom pathlib import Path\nimport argparse\nimport csv\n\nFIELD_NAMES = {\n    "M3C2 distance": "M3C2_distance", "distance uncertainty": "LoD95",\n    "significant change": "significant", "STD_cloud1": "STD_jan09",\n    "STD_cloud2": "STD_jun10", "Npoints_cloud1": "N_jan09",\n    "Npoints_cloud2": "N_jun10", "C2C absolute distances": "C2C_distance",\n}\n\n\n\ndef read_export(path: Path):\n    """Read comma, tab, semicolon, or whitespace exports, validating optional point counts.\n\n    Values remain strings. No data rows or missing values are removed.\n    """\n    with path.open(newline="", encoding="utf-8-sig") as source:\n        first = source.readline().strip().removeprefix("//").strip()\n        delimiter = next((sep for sep in (",", "\\t", ";") if sep in first), None)\n        parse = (lambda line: next(csv.reader([line], delimiter=delimiter))) if delimiter else str.split\n        fields = [field.strip() for field in parse(first)]\n        if fields[:3] != ["X", "Y", "Z"]:\n            raise ValueError("Expected a column-name header starting with X Y Z.")\n        normalized = [FIELD_NAMES.get(field.replace("_", " "), FIELD_NAMES.get(field, field)) for field in fields]\n        if len(set(normalized)) != len(normalized):\n            raise ValueError("Duplicate column names after normalization.")\n        rows = []\n        declared_count = None\n        for line_number, line in enumerate(source, start=2):\n            row = parse(line)\n            if line_number == 2 and len(row) == 1 and row[0].strip().isdigit():\n                declared_count = int(row[0])\n                continue\n            if len(row) != len(fields):\n                raise ValueError(f"Line {line_number}: field count differs from header; no rows were removed.")\n            rows.append(row)\n        if declared_count is not None and declared_count != len(rows):\n            raise ValueError(f"Point-count header says {declared_count}; found {len(rows)} data rows.")\n    return normalized, rows, declared_count\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("input", type=Path)\n    parser.add_argument("output", type=Path)\n    args = parser.parse_args()\n    if args.input.resolve() == args.output.resolve() or args.output.exists():\n        parser.error("Choose a new output file; existing files are preserved.")\n    fields, rows, declared_count = read_export(args.input)\n    with args.output.open("x", newline="") as target:\n        writer = csv.writer(target)\n        writer.writerow(fields)\n        writer.writerows(rows)\n    if declared_count is not None:\n        print(f"Validated optional point-count header: {declared_count}")\n    print(f"Preserved {len(rows)} rows in {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')


expected_fields = {
    "m3c2.asc": ("M3C2_distance", "LoD95", "significant"),
    "c2c.asc": ("C2C_distance",),
}
spec = importlib.util.spec_from_file_location("export_helper", helper_path)
converter = importlib.util.module_from_spec(spec)
spec.loader.exec_module(converter)
converted_files = []
for filename, expected in expected_fields.items():
    if filename not in uploaded or not Path(filename).is_file():
        raise ValueError("Run step 1 and upload both exports first.")
    header, rows, declared_count = converter.read_export(Path(filename))
    absent = [field for field in expected if field not in header]
    if absent:
        raise ValueError(
            filename + " is missing " + ", ".join(absent)
            + ". Check the selected cloud and ASCII export options."
        )
    output = output_folder / Path(filename).with_suffix(".csv").name
    result = subprocess.run(
        [sys.executable, str(helper_path), filename, str(output)],
        capture_output=True, text=True,
    )
    if result.returncode:
        raise RuntimeError(result.stderr.strip())
    print(output.name + ": Preserved " + str(len(rows)) + " rows")
    if declared_count is not None:
        print("Validated optional point-count header: " + str(declared_count))
    if filename == "c2c.asc" and len(rows) != 689560:
        print("CHECK C2C DIRECTION: this exercise expects 689560 June points. "
              "Confirm June was the compared cloud and January the reference. "
              "All uploaded rows have been preserved.")
    converted_files.append(output)

archive = output_folder / "exercise06_qgis_csvs.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for output in converted_files:
        bundle.write(output, arcname=output.name)
archive_path = archive
print("Conversion complete. Run step 3 to download both CSV files.")

## 3 Download your results

Run the cell below to download **exercise06_qgis_csvs.zip**. Extract
the ZIP on your computer: it contains **m3c2.csv** and **c2c.csv**.
Download the files before leaving Colab; runtime files are temporary.

Continue with Part 4 in QGIS. Import each CSV as Delimited Text with
**X field = X**, **Y field = Y**, and **EPSG:26913**. Check that the
coordinates are around **X = 386000 m** and **Y = 3632800 m**. If they
are small shifted coordinates, correct the CloudCompare export first;
this notebook does not change coordinates.

The M3C2 CSV uses **M3C2_distance**, **LoD95**, and **significant**.
LoD95 is the exported distance uncertainty, already a detection
threshold. The C2C CSV uses **C2C_distance**.

In [ ]:
if "archive_path" not in globals() or archive_path is None or not archive_path.is_file():
    raise RuntimeError("Complete step 2 successfully before downloading.")
files.download(str(archive_path))